# LongMemEval domain fine-tune (T4 Colab)

Self-contained: clones the experiment branch, downloads LongMemEval-s
from HuggingFace (~265 MB), runs the data prep + ingest, then trains
with `vstash retrain --training-queries ... --eval-queries ...`
(non-synthetic, labeled v5 recipe). No Drive upload needed.

Reported as a SEPARATE result from the LongMemEval baseline
retrieval numbers: this is a domain-adaptation demonstration
("training on chat data improves chat retrieval"), not a way to
inflate v3's headline number.

## GPU usage map

Training auto-uses CUDA via `SentenceTransformer.fit()` whenever
`torch.cuda.is_available()` is True (the cell-4 sanity check
verifies this).  The other heavy steps:

| Step | Where it runs | How it picks the device |
|---|---|---|
| Bulk-mine (corpus encode + matmul) | GPU | `--bulk-mine-device cuda` (explicit) |
| Train MNRL | GPU | SentenceTransformer auto-detect |
| Eval baseline + final | GPU | SentenceTransformer auto-detect |
| Ingest (cell 3, in `prepare_retrain`) | CPU | FastEmbed ONNX (no CUDA wheel) |

Cell 4 also exports `CUDA_VISIBLE_DEVICES=0` so accelerate / torch
do not silently fall back to CPU on a misconfigured runtime.

In [ ]:
# Cell 1: Setup -- clone the branch with the LongMemEval experiment
# scripts (lme_prepare_retrain, longmemeval_retrieval).  --eval-queries
# is on develop already (PR #299) so any develop-or-later branch works.
BRANCH = 'feature/longmemeval-retrain-experiments'

!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0' huggingface_hub
!rm -rf /content/vstash
!git clone --branch $BRANCH https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .
!vstash --version
!vstash retrain --help | grep -E '\-\-training-queries|\-\-eval-queries' | head -4

In [ ]:
# Cell 2: Download longmemeval_s (~265 MB) from HF directly into the
# experiments/data/longmemeval/ path that the prep script expects.
import os
from huggingface_hub import hf_hub_download

TARGET = '/content/vstash/experiments/data/longmemeval'
os.makedirs(TARGET, exist_ok=True)

path = hf_hub_download(
    repo_id='xiaowu0162/longmemeval',
    filename='longmemeval_s',
    repo_type='dataset',
    local_dir=TARGET,
)
print(f'Downloaded longmemeval_s ({os.path.getsize(path) / 1024 / 1024:.1f} MB) -> {path}')

In [ ]:
# Cell 3: Build the corpus DB + train/eval JSONLs.
# Stratified 80/20 split by question_type, deterministic seed=42.
# ~9 min on Colab CPU before the GPU phase begins (ingest is
# embedding-bound and FastEmbed has no CUDA wheel; the GPU phase
# is bulk_mine + train + eval).
import os
os.chdir('/content/vstash')

!python -m experiments.lme_prepare_retrain \
    --output-db    /content/lme_corpus.db \
    --output-train /content/lme_train.jsonl \
    --output-eval  /content/lme_eval.jsonl \
    --output-meta  /content/lme_retrain_meta.json \
    --force

import json
meta = json.load(open('/content/lme_retrain_meta.json'))
print()
print('Split:')
for k in ('n_train_questions', 'n_holdout_questions', 'n_train_qrels',
         'n_eval_qrels', 'n_train_docs', 'n_holdout_docs'):
    print(f'  {k:25s} {meta[k]}')
print('By type:')
for t, counts in meta['by_type'].items():
    print(f'  {t:30s} train={counts["train"]:3d}  holdout={counts["holdout"]:3d}')

In [ ]:
# Cell 4: Verify the ENTIRE retrain pipeline (mining + training +
# eval) will run on T4 GPU before we kick off the long retrain.
# The CLI exposes --bulk-mine-device for the miner only; train_mnrl
# and evaluate_model rely on SentenceTransformer's auto device
# selection, so we load one here and assert .device.type=='cuda'.
# If this assert ever fails the runtime is wrong and the long
# retrain would silently degrade to CPU.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['VSTASH_DB_PATH'] = '/content/lme_corpus.db'

import torch
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), (
    'CUDA is not available. Runtime -> Change runtime type -> T4 GPU.'
)
print('torch:', torch.__version__)
print('cuda device:', torch.cuda.get_device_name(0))
print('mem GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

_probe = SentenceTransformer('BAAI/bge-small-en-v1.5')
assert _probe.device.type == 'cuda', (
    f'SentenceTransformer landed on {_probe.device}; train will use CPU. '
    'Restart runtime and re-check.'
)
print('SentenceTransformer device:', _probe.device, '-- training will use GPU.')
del _probe; torch.cuda.empty_cache()

BASE_MODEL = 'BAAI/bge-small-en-v1.5'
OUTPUT_PATH = '/content/bge-small-rrf-lme-v1'
MAX_QUERIES = 5000
EPOCHS = 2
LR = 3e-6
BATCH = 64
SEED = 42
print('VSTASH_DB_PATH ->', os.environ['VSTASH_DB_PATH'])

In [ ]:
# Cell 5: Run vstash retrain.  All three GPU steps:
#   - generate_labeled_triples_batched: --bulk-mine-device cuda (explicit).
#   - train MNRL: SentenceTransformer auto-detects CUDA (verified in cell 4).
#   - evaluate_model baseline + final: same auto-detect.
# Gate refuses to save if NDCG@10 does not improve (--min-gain 0.0).
import time
t0 = time.perf_counter()
!vstash retrain \
    --training-queries /content/lme_train.jsonl \
    --eval-queries     /content/lme_eval.jsonl \
    --base-model       $BASE_MODEL \
    --output           $OUTPUT_PATH \
    --max-queries      $MAX_QUERIES \
    --epochs           $EPOCHS \
    --lr               $LR \
    --batch-size       $BATCH \
    --seed             $SEED \
    --bulk-mine        \
    --bulk-mine-device cuda
print(f'\n[retrain base BGE] wall: {time.perf_counter() - t0:.1f}s')

# Belt-and-suspenders: confirm the saved model loads on CUDA.
import torch
from sentence_transformers import SentenceTransformer
_check = SentenceTransformer(OUTPUT_PATH)
print('saved model device:', _check.device)
del _check; torch.cuda.empty_cache()

In [ ]:
# Cell 6: Persist the trained model to Drive so the local Mac can
# download it and run the full retrieval-side benchmark
# (experiments.longmemeval_retrieval --model <path>) against the
# same 500 questions used for the v3 baseline.
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/lme_retrain'
os.makedirs(DRIVE_OUT, exist_ok=True)

for suffix in ('', '.candidate', '.old'):
    src = OUTPUT_PATH + suffix
    if os.path.isdir(src):
        dst = os.path.join(DRIVE_OUT, f'bge-small-rrf-lme-v1{suffix}')
        !cp -r "$src" "$dst"
        print(f'Copied {src} -> {dst}')
    else:
        print(f'(skip) {src} does not exist')
!ls -la $DRIVE_OUT | grep lme-v1

In [ ]:
# Cell 7 (optional): try v3 as base model.  v3 is published as a
# SentenceTransformer-loadable repo on HF, so we can stack the
# domain fine-tune on top of it instead of starting from base BGE.
# The two arms answer different questions:
#   base BGE -> bge-small-rrf-lme-v1: pure 'chat data lifts BGE'.
#   v3       -> bge-small-rrf-lme-v1-from-v3: 'chat data on top of
#               BEIR-tuned weights still lifts further?'.
import time
OUTPUT_V3 = '/content/bge-small-rrf-lme-v1-from-v3'
BASE_V3   = 'Stffens/bge-small-rrf-v3'
t0 = time.perf_counter()
!vstash retrain \
    --training-queries /content/lme_train.jsonl \
    --eval-queries     /content/lme_eval.jsonl \
    --base-model       $BASE_V3 \
    --output           $OUTPUT_V3 \
    --max-queries      $MAX_QUERIES \
    --epochs           $EPOCHS \
    --lr               $LR \
    --batch-size       $BATCH \
    --seed             $SEED \
    --bulk-mine        \
    --bulk-mine-device cuda
print(f'\n[retrain v3-base] wall: {time.perf_counter() - t0:.1f}s')

import os
for suffix in ('', '.candidate'):
    src = OUTPUT_V3 + suffix
    if os.path.isdir(src):
        dst = os.path.join(DRIVE_OUT, f'bge-small-rrf-lme-v1-from-v3{suffix}')
        !cp -r "$src" "$dst"
        print(f'Copied {src} -> {dst}')